# Notebook 02: The Transformer Encoder

## Why This Matters

The transformer encoder is the backbone of BERT, RoBERTa, DeBERTa, and virtually every state-of-the-art text understanding model used in industry today. Search engines use encoder representations for semantic retrieval. Classification pipelines, NER systems, and sentence embedding models all depend on it. Understanding what each component does — and why — lets you debug training instabilities, choose the right normalization, and reason about why 24-layer BERT learns what it learns. This notebook builds a complete encoder from scratch, including the design decisions that separate research-grade from production-grade implementations.

## Background

The encoder takes a sequence of tokens and outputs a sequence of contextualized representations — each token's embedding now contains information from the entire sequence. It achieves this through repeated application of two sublayers:

1. **Multi-Head Self-Attention (MHA)**: every position looks at all others
2. **Position-wise Feed-Forward Network (FFN)**: a two-layer MLP applied identically to each position

Each sublayer is wrapped in a **residual connection** and followed by **layer normalization**. The question of whether LN comes before or after the sublayer (Pre-LN vs Post-LN) has significant implications for training stability, especially at scale.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
print("Ready.")

## 1. Residual Connections: The Gradient Highway

Every sublayer in the transformer follows this pattern:

$$x_{\text{out}} = \text{LayerNorm}(x + \text{Sublayer}(x))  \quad \text{(Post-LN)}$$

**Why residual connections?** They create a direct gradient highway from the loss back to early layers. Without them, gradients must pass through every transformation, shrinking exponentially (vanishing gradient). With residuals, the gradient of the loss with respect to $x$ is:

$$\frac{\partial \mathcal{L}}{\partial x} = \frac{\partial \mathcal{L}}{\partial x_{\text{out}}} \cdot \left(1 + \frac{\partial \text{Sublayer}(x)}{\partial x}\right)$$

The "$1$" term means the gradient always has a direct path back, regardless of what the sublayer does. This is why ResNets and Transformers can be trained to 100+ layers.

**Intuition:** The network learns **residuals** (corrections) rather than full transformations. Early in training, sublayers learn small corrections to the identity function.

## 2. Layer Normalization

Layer Norm normalizes across the **feature dimension** for each token independently:

$$\text{LayerNorm}(x) = \frac{x - \mu}{\sigma + \epsilon} \cdot \gamma + \beta$$

where $\mu$ and $\sigma$ are computed over the $d_{\text{model}}$ features of a single token, and $\gamma, \beta$ are learned affine parameters (per-feature scale and shift).

**Why not BatchNorm?** BatchNorm normalizes across the batch dimension. For sequences, (a) batch sizes can be small, making batch statistics noisy, and (b) at inference with batch size 1, batch statistics are undefined. LayerNorm uses per-sample statistics, making it batch-size independent — essential for autoregressive decoding.

**Interview question:** "What does LayerNorm actually normalize?"  
**Answer:** It normalizes the activations across the embedding dimension for each token independently. It does NOT normalize across time or batch. This means each token's 512-dimensional vector is rescaled to zero mean, unit variance, then learned scale/shift is applied.

In [ ]:
class LayerNorm(nn.Module):
    """Manual LayerNorm to see exactly what's happening."""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, x):
        # x: (B, T, d_model) — normalize over last dim
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True, unbiased=False)
        x_norm = (x - mean) / (std + self.eps)
        return self.gamma * x_norm + self.beta

# Verify against PyTorch's implementation
d_model = 128
ln_manual = LayerNorm(d_model)
ln_torch = nn.LayerNorm(d_model)

x = torch.randn(4, 10, d_model) * 5 + 3  # Offset to make normalization visible
out_manual = ln_manual(x)
out_torch = ln_torch(x)

print(f"Manual LN output mean: {out_manual.mean().item():.6f}  (should be ~0)")
print(f"Manual LN output std:  {out_manual.std().item():.6f}  (should be ~1)")
print(f"Max diff vs torch LN:  {(out_manual - out_torch).abs().max().item():.8f}")

## 3. Pre-LN vs Post-LN: A Critical Design Choice

**Original Transformer (Post-LN):**
$$x \leftarrow \text{LN}(x + \text{MHA}(x))$$

**Modern Transformers (Pre-LN):**
$$x \leftarrow x + \text{MHA}(\text{LN}(x))$$

**Why Pre-LN wins for deep networks:**

Post-LN was the original design and works for shallow models. But for deep models (12+ layers), Post-LN causes a training instability: the residual stream grows in scale across layers because LN is applied *after* the residual addition. The gradient magnitude also varies dramatically across layers, requiring very careful learning rate warmup.

Pre-LN normalizes the input *before* each sublayer. This keeps the residual stream in a stable scale range throughout training — LN is now applied to the input $x$ directly, which remains well-scaled because it's passed through by the residual. Pre-LN allows much more aggressive learning rates and removes the need for careful warmup.

**Used in:** GPT-2, GPT-3, LLaMA, Mistral, Gemma, and virtually all modern LLMs use Pre-LN (often with RMSNorm instead of LayerNorm).

## 4. Feed-Forward Network: The 4× Expansion

Each transformer layer contains a position-wise FFN:

$$\text{FFN}(x) = W_2 \cdot \text{GELU}(W_1 x + b_1) + b_2$$

where $W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$ and $d_{\text{ff}} = 4 \times d_{\text{model}}$ by convention.

**Why 4× expansion?** The FFN expands into a higher-dimensional space where nonlinear transformations are applied, then projects back. The intermediate dimension needs to be large enough to represent complex feature interactions. Empirically, 4× was found to work well in the original paper and became the standard. Some modern models use different ratios (e.g., 8/3 for SwiGLU variants).

**The FFN as memory (Geva et al. 2021):** Research shows that FFN layers act as key-value memories. The first layer creates "keys" that match input patterns; the second layer retrieves associated "values" (factual associations, world knowledge). This is why distilling factual knowledge into a model requires large FFN layers.

**GELU vs ReLU:** GELU (Gaussian Error Linear Unit) is smoother than ReLU and empirically improves performance. Mathematically: $\text{GELU}(x) = x \cdot \Phi(x)$ where $\Phi$ is the Gaussian CDF. It has a small amount of "leakage" for negative values, which helps gradient flow.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1, activation='gelu'):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.gelu if activation == 'gelu' else F.relu

    def forward(self, x):
        return self.linear2(self.dropout(self.activation(self.linear1(x))))

# Parameter count
d_model = 512
ffn = FeedForward(d_model)
ffn_params = sum(p.numel() for p in ffn.parameters())
print(f"FFN parameters (d_model={d_model}, d_ff={4*d_model}): {ffn_params:,}")
print(f"  = 2 * {d_model} * {4*d_model} + {d_model} + {4*d_model}")
print(f"  = {2 * d_model * 4*d_model + d_model + 4*d_model:,} (without biases: {2*d_model*(4*d_model):,})")

# Compare FFN vs MHA parameter counts
mha_params = 4 * d_model ** 2  # Q, K, V, O
ffn_params_no_bias = 2 * d_model * (4 * d_model)
print(f"\nMHA params:  {mha_params:,}")
print(f"FFN params:  {ffn_params_no_bias:,}")
print(f"FFN/MHA ratio: {ffn_params_no_bias/mha_params:.1f}x")
print("(In modern LLMs, FFN typically holds ~2/3 of total parameters)")

## 5. Building the Encoder Layer (Pre-LN)

Putting it together: each encoder layer applies self-attention and FFN with Pre-LN and residual connections.

In [ ]:
# Re-use MHA from notebook 01 (copy here for self-containment)
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V), attn_weights

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B, T_q, _ = query.shape
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))
        attn_out, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T_q, self.d_model)
        return self.W_o(attn_out), attn_weights

class TransformerEncoderLayer(nn.Module):
    """Pre-LN transformer encoder layer."""
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_key_padding_mask=None):
        # Pre-LN: normalize BEFORE sublayer
        residual = x
        x = self.norm1(x)
        attn_out, _ = self.self_attn(x, x, x, mask=src_key_padding_mask)
        x = residual + self.dropout(attn_out)

        residual = x
        x = self.norm2(x)
        ffn_out = self.ffn(x)
        x = residual + self.dropout(ffn_out)
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff=None,
                 max_len=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)  # Final LN (Pre-LN style)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_key_padding_mask=None):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        x = self.dropout(self.embedding(x) + self.pos_embedding(positions))
        for layer in self.layers:
            x = layer(x, src_key_padding_mask)
        return self.norm(x)

# Build and test
vocab_size, d_model, n_heads, n_layers = 1000, 128, 8, 4
encoder = TransformerEncoder(vocab_size, d_model, n_heads, n_layers).to(device)
tokens = torch.randint(0, vocab_size, (2, 16)).to(device)
enc_out = encoder(tokens)
print(f"Encoder output shape: {enc_out.shape}")  # (2, 16, 128)

total_params = sum(p.numel() for p in encoder.parameters())
print(f"Total encoder parameters: {total_params:,}")

## 6. CLS Token Pooling for Classification

BERT introduced the [CLS] (classification) token prepended to every input. After passing through all encoder layers, the [CLS] token's output representation aggregates information from the entire sequence via attention, making it a natural sequence-level representation for classification tasks.

**Why CLS and not mean pooling?** Both work in practice. CLS is learned to be informative for classification through the pre-training objective (NSP in BERT). Mean pooling over all tokens is often slightly better for sentence embeddings (SimCSE paper shows this). Many modern embedding models (E5, GTE, NV-Embed) use mean pooling.

In [ ]:
class EncoderForClassification(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, n_classes,
                 dropout=0.1):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, d_model, n_heads, n_layers,
                                          dropout=dropout)
        self.cls_token_id = vocab_size - 1  # Use last vocab id as [CLS]
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, input_ids):
        B, T = input_ids.shape
        # Prepend CLS token
        cls_tokens = torch.full((B, 1), self.cls_token_id,
                                device=input_ids.device)
        x = torch.cat([cls_tokens, input_ids], dim=1)  # (B, T+1)
        enc_out = self.encoder(x)        # (B, T+1, d_model)
        cls_repr = enc_out[:, 0, :]     # (B, d_model) — CLS position
        return self.classifier(cls_repr)

model = EncoderForClassification(1000, 128, 8, 4, n_classes=5).to(device)
tokens = torch.randint(0, 999, (4, 20)).to(device)
logits = model(tokens)
print(f"Classification logits shape: {logits.shape}")   # (4, 5)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. BERT-Style Masked Language Modeling (Conceptual + Sketch)

BERT's pre-training objective is **Masked Language Modeling (MLM)**:
1. Randomly mask 15% of input tokens (replace with [MASK] token)
2. Train the model to predict the original token at each masked position
3. This forces the model to learn bidirectional context — to predict a masked word, it must understand its entire surrounding context

**The 80/10/10 rule:** Of the 15% selected tokens:
- 80% are replaced with [MASK]
- 10% are replaced with a random token (forces robustness)
- 10% are left unchanged (keeps representations meaningful for downstream tasks)

**Why not just predict all tokens?** This would be autoregressive (GPT-style). MLM allows the model to see context from both directions simultaneously, producing better representations for understanding tasks (classification, NER, QA) at the cost of not being generative.

In [ ]:
def create_mlm_inputs(token_ids, vocab_size, mask_token_id, mask_prob=0.15):
    """
    Create MLM inputs following BERT's 80/10/10 strategy.
    Returns: (masked_ids, labels) where labels=-100 for unmasked positions.
    """
    labels = token_ids.clone()
    # Select 15% of positions
    probability_matrix = torch.full(token_ids.shape, mask_prob)
    masked_indices = torch.bernoulli(probability_matrix).bool()

    # -100 is the ignore index for CrossEntropyLoss
    labels[~masked_indices] = -100

    # 80%: replace with [MASK]
    indices_replaced = torch.bernoulli(torch.full(token_ids.shape, 0.8)).bool() & masked_indices
    token_ids[indices_replaced] = mask_token_id

    # 10%: replace with random token
    indices_random = (torch.bernoulli(torch.full(token_ids.shape, 0.5)).bool()
                      & masked_indices & ~indices_replaced)
    random_words = torch.randint(vocab_size, token_ids.shape)
    token_ids[indices_random] = random_words[indices_random]
    # remaining 10%: unchanged

    return token_ids, labels

# Sketch of MLM head
class MLMHead(nn.Module):
    def __init__(self, d_model, vocab_size):
        super().__init__()
        self.dense = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.decoder = nn.Linear(d_model, vocab_size)

    def forward(self, hidden_states):
        x = self.dense(hidden_states)
        x = F.gelu(x)
        x = self.norm(x)
        return self.decoder(x)  # (B, T, vocab_size)

vocab_size = 1000
tokens = torch.randint(10, vocab_size - 10, (2, 20))
masked_tokens, labels = create_mlm_inputs(tokens.clone(), vocab_size,
                                           mask_token_id=1)
print(f"Masked positions: {(labels != -100).sum().item()} of {tokens.numel()}")
print(f"Fraction masked: {(labels != -100).float().mean().item():.2f} (target ~0.15)")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Encoder function | Maps token sequence → contextualized representations (all positions see all others) |
| Residual connection | Gradient highway: $x + \text{sublayer}(x)$; enables training of deep networks |
| Post-LN | Original design: LN after residual; unstable for deep models, needs warmup |
| Pre-LN | Modern standard: LN before sublayer; stable training, used in all major LLMs |
| LayerNorm | Normalizes over $d_{\text{model}}$ features per token; batch-size independent |
| FFN 4× expansion | $d_{\text{ff}} = 4 \times d_{\text{model}}$; ~2/3 of model params live in FFN layers |
| GELU activation | Smoother than ReLU; empirically better; $x \cdot \Phi(x)$ |
| CLS token | Prepended token that aggregates sequence representation via attention |
| MLM pre-training | Predict randomly masked tokens; enables bidirectional context modeling |
| Encoder param count | Per layer: $4d^2$ (MHA) + $8d^2$ (FFN) $= 12d^2$; BERT-base: $\sim 85M$ |